# CelebrIA -  Descubre a tu famoso más parecido




Instalamos las librerías necesarias

In [ ]:
!pip install --quiet facenet-pytorch==2.5.2 torchvision pillow numpy opencv-python


Importamos las librerías del proyecto

In [ ]:
from IPython.display import display, HTML
from google.colab.output import eval_js
from base64 import b64decode
import os

import torch
from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image
import numpy as np
import json
from datetime import datetime
import cv2
from matplotlib import pyplot as plt
import matplotlib.image as mpimg


import zipfile
import torch
import requests

import tempfile
import kagglehub

import random

Descargamos el dataset de famosos

In [ ]:
# Download latest version
dataset_path = kagglehub.dataset_download("vishesh1412/celebrity-face-image-dataset")

print("Path to dataset files:", dataset_path)

Parámetros de configuración

In [ ]:
REF_EMBED_PATHS = ['/content/embedding.npy']  # ruta al embedding de la Celda 3
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}

Funciones

In [ ]:
def take_photo(filename='photo.jpg', quality=0.8):
    js_code = f"""
    async function takePhoto(quality) {{
      const div = document.createElement('div');
      const video = document.createElement('video');
      const button = document.createElement('button');
      button.textContent = 'Capturar';
      div.appendChild(video);
      div.appendChild(button);
      document.body.appendChild(div);

      const stream = await navigator.mediaDevices.getUserMedia({{video: true}});
      video.srcObject = stream;
      await video.play();

      // Espera a que el usuario pulse el botón
      await new Promise((resolve) => button.onclick = resolve);

      // Ajustar canvas al tamaño del video
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getTracks().forEach(track => track.stop());
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }}
    takePhoto({quality})
    """
    # eval_js recibe un string con el código JS (no el objeto Javascript)
    data = eval_js(js_code)
    if not data:
        raise RuntimeError("No se obtuvo data desde la cámara.")
    header, encoded = data.split(',', 1)
    binary = b64decode(encoded)
    path = os.path.join('/content', filename)
    with open(path, 'wb') as f:
        f.write(binary)
    return path

def get_embedding(image_path, save_npy='/content/embedding.npy', save_json='/content/embedding.json'):
    # cargar imagen
    img = Image.open(image_path).convert('RGB')

    # detectar cajas y probabilidades
    boxes, probs = mtcnn.detect(img)
    if boxes is None or len(boxes) == 0:
        raise RuntimeError("No se detectaron caras en la imagen.")

    # obtener tensores recortados normalizados
    faces = mtcnn(img)  # devuelve (N,3,160,160) o (3,160,160) si solo 1
    if isinstance(faces, torch.Tensor) and faces.dim() == 3:
        faces = faces.unsqueeze(0)

    # seleccionar cara más grande (opcional)
    areas = [ ( (b[2]-b[0])*(b[3]-b[1]) , i ) for i,b in enumerate(boxes) ]
    areas.sort(reverse=True)
    idx = areas[0][1]

    face_tensor = faces[idx].unsqueeze(0).to(device)  # (1,3,160,160)

    # obtener embedding
    with torch.no_grad():
        embedding = resnet(face_tensor).cpu().numpy().reshape(-1)  # (512,)

    # dibujar bbox para feedback y mostrar
    img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    x1, y1, x2, y2 = map(int, boxes[idx])
    cv2.rectangle(img_cv, (x1,y1), (x2,y2), (0,255,0), 2)
    cv2.putText(img_cv, "Cara", (x1, max(y1-10,10)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
    # mostrar con matplotlib (colores RGB)
    plt.figure(figsize=(6,6))
    plt.axis('off')
    plt.imshow(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))
    plt.show()

    # guardar embedding
    np.save(save_npy, embedding)
    record = {
        "created_at": datetime.utcnow().isoformat() + "Z",
        "embedding_length": int(embedding.shape[0]),
        "embedding": embedding.tolist(),
        "face_box": {"x1": int(x1), "y1": int(y1), "x2": int(x2), "y2": int(y2)},
        "device": str(device)
    }
    with open(save_json, 'w', encoding='utf-8') as f:
        json.dump(record, f, indent=2, ensure_ascii=False)

    print(f"Embedding guardado en {save_npy} (shape {embedding.shape})")
    print(f"Metadatos guardados en {save_json}")
    print("Primeras 10 componentes:", ", ".join(f"{float(x):.6f}" for x in embedding[:10]))
    return embedding

def find_folders(root_dir):
    items = []
    for entry in sorted(os.listdir(root_dir)):
        full = os.path.join(root_dir, entry)
        if os.path.isdir(full):
            items.append(full)
    return items

def find_image_files(folder):
    imgs = []
    for dirpath, _, filenames in os.walk(folder):
        for fn in filenames:
            if os.path.splitext(fn.lower())[1] in IMAGE_EXTS:
                imgs.append(os.path.join(dirpath, fn))
    return imgs

def choose_representative_image(img_paths):
    if not img_paths:
        return None
    img_paths = sorted(img_paths)
    img_paths.sort(key=lambda p: os.path.getsize(p), reverse=True)
    return img_paths[0]

def load_reference_embedding(possible_paths):
    for p in possible_paths:
        if os.path.exists(p):
            emb = np.load(p)
            emb = emb.reshape(-1)
            print(f"Embedding de referencia cargado desde {p} (len={len(emb)})")
            return emb
    raise FileNotFoundError("No se encontró el embedding de referencia. Asegúrate de que la Celda 3 guardó '/content/embedding.npy'.")

def l2_norm(x):
    return np.linalg.norm(x)

def cosine_similarity(a, b):
    na = l2_norm(a)
    nb = l2_norm(b)
    if na == 0 or nb == 0:
        return -1.0
    return float(np.dot(a, b) / (na * nb))

def prettify_name_from_folder(folder_path):
    # Extrae el nombre desde el nombre de la carpeta y normaliza:
    #name = os.path.basename(folder_path)
    name = os.path.basename(os.path.dirname(folder_path))
    name = name.replace('_', ' ').replace('-', ' ').strip()
    # Capitalizar cada palabra
    name = " ".join(w.capitalize() for w in name.split())
    return name

----------

Primero capturamos la foto del usuario

In [ ]:
path = take_photo('photo.jpg')

img = mpimg.imread("photo.jpg")
plt.imshow(img)
plt.axis('off')  # Oculta los ejes
plt.show()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# crear modelos
mtcnn = MTCNN(keep_all=True, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

Calculamos el *embedding* desde la foto guardada


In [ ]:
emb = get_embedding(path)
print(emb)

Comparar embeddings de un ZIP y mostrar la foto más similar

In [ ]:
# --- Cargar embedding de referencia ---
ref_emb = load_reference_embedding(REF_EMBED_PATHS)

# --- 4. Iterar carpetas, elegir imagen por carpeta, calcular embedding y comparar ---
#folders = find_folders(EXTRACT_DIR)
folders = find_folders(dataset_path)
if not folders:
    print("No se encontraron carpetas en el ZIP. Asegúrate de que el ZIP contiene carpetas con imágenes.")
else:
    results = []  # lista de dicts {folder, img_path, similarity}
    folder = folders[0]

    img_files = find_image_files(folder)
    for i in range(0,30):
        img_index= random.randint(0,len(img_files))
        chosen = img_files[img_index] #choose_representative_image(img_files)
        if chosen is None:
            print(f"[SKIP] No se pudo elegir imagen en: {folder}")
            continue

        # cargar imagen y detectar caras
        img_pil = Image.open(chosen).convert('RGB')
        boxes, probs = mtcnn.detect(img_pil)
        if boxes is None or len(boxes) == 0:
            print(f"[SKIP] No se detectó cara en la imagen elegida: {chosen}")
            continue

        faces = mtcnn(img_pil)
        if isinstance(faces, torch.Tensor) and faces.dim() == 3:
            faces = faces.unsqueeze(0)

        # seleccionar la cara más grande
        areas = [ ( (b[2]-b[0])*(b[3]-b[1]) , i ) for i,b in enumerate(boxes) ]
        areas.sort(reverse=True)
        idx = areas[0][1]

        face_tensor = faces[idx].unsqueeze(0).to(device)
        with torch.no_grad():
            emb = resnet(face_tensor).cpu().numpy().reshape(-1)

        sim = cosine_similarity(ref_emb, emb)
        results.append({
            'folder': folder,
            'img': chosen,
            'embedding': emb,
            'similarity': sim
        })
        celebrity_name= prettify_name_from_folder(chosen)
        print(f"Famoso: {celebrity_name} | Similitud: {sim:.4f}")

    if not results:
        print("No se obtuvieron embeddings válidos de ninguna carpeta (quizá no había caras detectables).")
    else:
        # ordenar por similitud descendente
        results.sort(key=lambda r: r['similarity'], reverse=True)
        best = results[0]
        best_name = prettify_name_from_folder(best['img'])
        print("\nMejor coincidencia:")
        print(f"Imagen: {best['img']}")
        print(f"Nombre extraído: {best_name}")
        print(f"Similitud cosine: {best['similarity']:.6f}")

        # mostrar imagen de referencia y mejor coincidencia lado a lado,
        # con el nombre del famoso encima de la foto de la derecha
        ref_img_path = '/content/photo.jpg'  # la foto tomada en la Celda 2 (si existe)
        fig = plt.figure(figsize=(10,5))

        # Subplot 1: referencia si existe
        ax1 = fig.add_subplot(1,2,1)
        if os.path.exists(ref_img_path):
            img_ref = Image.open(ref_img_path).convert('RGB')
            ax1.imshow(img_ref)
            ax1.set_title("Tú")
        else:
            ax1.text(0.5, 0.5, "Referencia: /content/photo.jpg\n(no encontrada)", ha='center', va='center', fontsize=12)
        ax1.axis('off')

        # Subplot 2: mejor coincidencia con nombre encima
        ax2 = fig.add_subplot(1,2,2)
        img_best = Image.open(best['img']).convert('RGB')
        ax2.imshow(img_best)
        # Mostrar el nombre del famoso como título (encima de la imagen)
        ax2.set_title(best_name + f"\n(sim={best['similarity']:.4f})", fontsize=14)
        ax2.axis('off')

        plt.show()

if os.path.exists("/content/photo.jpg"):
    os.remove("/content/photo.jpg")
    print("photo.jpg eliminado correctamente.")
else:
    print("El archivo no existe.")


Por último borramos la foto del usuario